In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Colab \Notebooks/microscopy_self_supervised_learning/

Imports & setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from skimage.io import imread
from skimage.transform import resize
import cv2

Paths & metadata

In [ ]:
metadata = pd.read_csv("data/processed/metadata_with_moa.csv")
metadata.head(3)

In [ ]:
DATA_DIR = "data/raw"
RESULTS_DIR = "results/figures"

In [ ]:
metadata['Image_FileName_DAPI'][0]

Image loader function

In [ ]:
import os
import tifffile as tiff
import numpy as np

BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/microscopy_self_supervised_learning/data/raw"

def fix_path(pathname):
    return pathname.split("/")[-1]

def load_3channel_image(row):
    paths = [
        os.path.join(BASE_DIR, fix_path(row["Image_PathName_DAPI"]), row["Image_FileName_DAPI"]),
        os.path.join(BASE_DIR, fix_path(row["Image_PathName_Tubulin"]), row["Image_FileName_Tubulin"]),
        os.path.join(BASE_DIR, fix_path(row["Image_PathName_Actin"]), row["Image_FileName_Actin"])
    ]

    channels = []
    for p in paths:
        img = tiff.imread(p).astype(np.float32)
        # normalize
        img = img / 65535.0
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)

        channels.append(img)

    img_stack = np.stack(channels, axis=0)   # (3, H, W)
    return img_stack

Dataset class

In [ ]:
from skimage.transform import resize
import torch

class MicroscopyDataset(Dataset):
    def __init__(self, metadata_df, img_size=128):
        self.meta = metadata_df.reset_index(drop=True)
        self.img_size = img_size

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        img = load_3channel_image(row)   # (3,1024,1280)

        # resize to (3,128,128)
        img = resize(img, (3, self.img_size, self.img_size), anti_aliasing=True)

        img = torch.tensor(img, dtype=torch.float32)
        label = row["moa"]
        return img, label

In [ ]:
path = '/content/drive/MyDrive/Colab Notebooks/microscopy_self_supervised_learning/data/raw/Week1_22123/Week1_150607_B04_s3_w135D66B4C-0548-4AB8-A57B-9CC39666813B.tif'
img = tiff.imread(path)

print(img.shape)
print(img.dtype)
print(img.min(), img.max())

DataLoader

In [ ]:
n = min(2000, len(metadata))
subset = metadata.sample(n=n, random_state=42)

In [ ]:
len(metadata)

In [ ]:
subset = metadata.sample(n=min(500, len(metadata)), random_state=42)
dataset = MicroscopyDataset(subset)

dataset = MicroscopyDataset(subset, img_size=128)
x, y = dataset[0]
print(x.shape)

In [ ]:
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
batch = next(iter(dataloader))
print(batch[0].shape)

Autoencoder model

In [ ]:
import torch
import torch.nn as nn

class Autoencoder(nn.Module):

    def __init__(self, latent_dim=128):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3,16,3,stride=2,padding=1),   # 3x128x128 → 16x64x64
            nn.ReLU(),

            nn.Conv2d(16,32,3,stride=2,padding=1),  # 16x64x64 → 32x32x32
            nn.ReLU(),

            nn.Conv2d(32,64,3,stride=2,padding=1),  # 32x32x32 → 64x16x16
            nn.ReLU(),

            nn.Flatten(),
            nn.Linear(64*16*16, latent_dim)
        )

        # Decoder
        self.decoder_fc = nn.Linear(latent_dim, 64*16*16)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64,32,3,stride=2,padding=1,output_padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(32,16,3,stride=2,padding=1,output_padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(16,3,3,stride=2,padding=1,output_padding=1),
            nn.Sigmoid()
        )

    def forward(self,x):

        z = self.encoder(x)

        x = self.decoder_fc(z)
        x = x.view(-1,64,16,16)

        out = self.decoder(x)

        return out, z

Train setup

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
model = Autoencoder(latent_dim=128).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 20

Training loop

In [ ]:
x, y = dataset[0]
print(x.shape)   # must be torch.Size([3,128,128])

In [ ]:
batch = next(iter(dataloader))
print(batch[0].shape)  # torch.Size([B,3,128,128])

In [ ]:
train_losses = []

for epoch in range(num_epochs):

    model.train()
    running_loss = 0.0

    loop = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for imgs, labels in loop:

        imgs = imgs.float().to(device)

        recon, embedding = model(imgs)

        loss = criterion(recon, imgs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        loop.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(dataloader)
    train_losses.append(epoch_loss)

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

    # ----- check reconstruction range -----
    model.eval()
    with torch.no_grad():

        sample_imgs = imgs[:6]            # take a few images from last batch
        sample_recon, _ = model(sample_imgs)

        print("Recon min:", sample_recon.min().item(),
              "Recon max:", sample_recon.max().item())

        # move to cpu for visualization later
        recon_examples = sample_recon.cpu()
        input_examples = sample_imgs.cpu()

In [ ]:
print(recon.min(), recon.max())

Plot training loss

In [ ]:
plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Autoencoder Training Loss")
plt.savefig(f"{RESULTS_DIR}/autoencoder_loss.png")
plt.show()

Reconstruction visualization

In [ ]:
fig, axes = plt.subplots(2,6, figsize=(12,4))

for i in range(6):

    axes[0,i].imshow(input_examples[i].permute(1,2,0))
    axes[0,i].axis("off")

    axes[1,i].imshow(recon_examples[i].permute(1,2,0))
    axes[1,i].axis("off")

axes[0,0].set_title("Original")
axes[1,0].set_title("Reconstructed")

plt.show()

In [ ]:
model.eval()
imgs, labels = next(iter(dataloader))
imgs = imgs.to(device)

with torch.no_grad():
    recon, z = model(imgs)

imgs = imgs.cpu().numpy()
recon = recon.cpu().numpy()

fig, axes = plt.subplots(3, 6, figsize=(12, 6))

for i in range(6):
    orig = imgs[i,0]
    rec  = recon[i,0]
    err  = np.abs(orig - rec)

    axes[0,i].imshow(orig, cmap="gray", vmin=0, vmax=1)
    axes[1,i].imshow(rec,  cmap="gray", vmin=0, vmax=1)
    axes[2,i].imshow(err,  cmap="hot")

    for j in range(3):
        axes[j,i].axis("off")

plt.suptitle("Top: Original | Middle: Reconstructed | Bottom: Error")
plt.savefig(f"{RESULTS_DIR}/autoencoder_reconstructions.png")
plt.show()

Flatten in encoder is problem Main problem: Flattening destroys spatial structure

Second problem: Huge compression immediately

Your first layer compresses:

49,152 features → 256

That is a 192× compression in one step.

Then you compress again:

256 → 128

So the encoder forces the entire image into 128 numbers almost instantly.

That is extremely aggressive and makes training unstable.

Second problem: Huge compression immediately

Your first layer compresses:

49,152 features → 256

That is a 192× compression in one step.

Then you compress again:

256 → 128

So the encoder forces the entire image into 128 numbers almost instantly.

That is extremely aggressive and makes training unstable.

In [ ]:
print(imgs.min(), imgs.max())

fixed : normalisation

Middle row (reconstruction):
❌ Looks like random noise / static
❌ No cell structure recovered
❌ Almost constant texture across images

→ confirms the model is not reconstructing meaningful signal.

Extract embeddings

In [ ]:
embeddings = []
labels = []

model.eval()

for _, row in tqdm(subset.iterrows(), total=len(subset)):

    img = load_3channel_image(row)  # (3,1024,1280)

    img_small = resize(img, (3,128,128), anti_aliasing=True)

    x = torch.tensor(img_small, dtype=torch.float32)
    x = x.unsqueeze(0).to(device)
    with torch.no_grad():
        recon, z = model(x)

    embeddings.append(z.cpu().numpy().squeeze())
    labels.append(row["moa"])

Save embeddings

In [ ]:
embeddings = np.array(embeddings)

np.save("data/processed/autoencoder_embeddings.npy", embeddings)
subset[["moa"]].to_csv("data/processed/autoencoder_labels.csv", index=False)

print("Saved embeddings:", embeddings.shape)

UMAP visualization

In [ ]:
from umap import UMAP
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(labels)

umap = UMAP(n_components=2, random_state=42)
Z = umap.fit_transform(embeddings)

plt.figure(figsize=(6,5))
plt.scatter(Z[:,0], Z[:,1], c=y_enc, cmap="tab10", s=10)
plt.title("Autoencoder embeddings (UMAP)")
plt.colorbar()
plt.savefig(f"{RESULTS_DIR}/umap_autoencoder_by_moa.png")
plt.show()